# 🤖 Notebook 4 — Modeling
**Project:** FAANG Stock Price Forecasting
**Goal:** Train 2 models on each stock — Linear Regression (baseline) and LSTM (deep learning).

> ⚠️ LSTM training may take 3–5 minutes. Run all cells in order.

## Step 1 — Install & Import Libraries

In [1]:
# Install tensorflow if not already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow', '--quiet'])
print('✅ TensorFlow install check done')

✅ TensorFlow install check done


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('✅ Libraries ready')
print(f'TensorFlow version: {tf.__version__}')

✅ Libraries ready
TensorFlow version: 2.21.0


## Step 2 — Load Processed Data

In [3]:
TICKERS  = ["META", "AAPL", "AMZN", "NFLX", "GOOGL"]
FEATURES = ["MA7", "MA21", "RSI", "Lag_1", "Lag_2", "Lag_3", "Volatility"]
TARGET   = "Close"
SEQ_LEN  = 60

train_data = {}
test_data  = {}

for ticker in TICKERS:
    train_data[ticker] = pd.read_csv(f"data/processed/{ticker}_train.csv", index_col="Date", parse_dates=True)
    test_data[ticker]  = pd.read_csv(f"data/processed/{ticker}_test.csv",  index_col="Date", parse_dates=True)
    print(f"  {ticker}: train={len(train_data[ticker])} rows | test={len(test_data[ticker])} rows")

print("\n✅ Data loaded for all 5 stocks")

  META: train=1190 rows | test=298 rows
  AAPL: train=1190 rows | test=298 rows
  AMZN: train=1190 rows | test=298 rows
  NFLX: train=1190 rows | test=298 rows
  GOOGL: train=1190 rows | test=298 rows

✅ Data loaded for all 5 stocks


## Step 3 — Model 1: Linear Regression (Baseline)

Simple model — uses our engineered features to predict closing price.

In [4]:
lr_predictions = {}

for ticker in TICKERS:
    X_train = train_data[ticker][FEATURES]
    y_train = train_data[ticker][TARGET]
    X_test  = test_data[ticker][FEATURES]

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    lr_predictions[ticker] = preds

    rmse = np.sqrt(mean_squared_error(test_data[ticker][TARGET], preds))
    mae  = mean_absolute_error(test_data[ticker][TARGET], preds)
    print(f"{ticker:5s} → RMSE: {rmse:.2f} | MAE: {mae:.2f}")

print("\n✅ Linear Regression done for all stocks")

META  → RMSE: 9.37 | MAE: 6.34
AAPL  → RMSE: 2.29 | MAE: 1.69
AMZN  → RMSE: 2.84 | MAE: 2.16
NFLX  → RMSE: 0.99 | MAE: 0.73
GOOGL → RMSE: 2.42 | MAE: 1.66

✅ Linear Regression done for all stocks


## Step 4 — Helper Function: Build LSTM Sequences

LSTM needs sequences of 60 days to predict the next day.

In [5]:
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i - seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

print("✅ Sequence builder ready")

✅ Sequence builder ready


## Step 5 — Model 2: LSTM (Deep Learning)

> This loop trains LSTM for each of the 5 stocks. Each stock takes ~1 minute.

In [6]:
lstm_predictions = {}
lstm_scalers     = {}

for ticker in TICKERS:
    print(f"\nTraining LSTM for {ticker}...")

    # Scale Close price to 0-1 range
    scaler      = MinMaxScaler(feature_range=(0, 1))
    train_close = train_data[ticker][[TARGET]].values
    test_close  = test_data[ticker][[TARGET]].values

    train_scaled = scaler.fit_transform(train_close)
    test_scaled  = scaler.transform(test_close)

    # Build sequences
    X_train, y_train = create_sequences(train_scaled, SEQ_LEN)
    combined         = np.concatenate([train_scaled, test_scaled])
    test_input       = combined[-(len(test_scaled) + SEQ_LEN):]
    X_test, _        = create_sequences(test_input, SEQ_LEN)

    # Reshape for LSTM [samples, timesteps, features]
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test  = X_test.reshape((X_test.shape[0],  X_test.shape[1],  1))

    # Build LSTM model
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")

    # Train with early stopping
    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.1,
        callbacks=[es],
        verbose=0
    )

    # Predict & inverse transform back to original price range
    preds_scaled = model.predict(X_test, verbose=0)
    preds        = scaler.inverse_transform(preds_scaled).flatten()

    lstm_predictions[ticker] = preds
    lstm_scalers[ticker]     = scaler

    actual = test_data[ticker][TARGET].values[-len(preds):]
    rmse   = np.sqrt(mean_squared_error(actual, preds))
    mae    = mean_absolute_error(actual, preds)
    print(f"  ✅ {ticker} → RMSE: {rmse:.2f} | MAE: {mae:.2f}")

print("\n✅ LSTM training complete for all stocks!")


Training LSTM for META...
  ✅ META → RMSE: 114.17 | MAE: 101.14

Training LSTM for AAPL...
  ✅ AAPL → RMSE: 12.32 | MAE: 9.97

Training LSTM for AMZN...
  ✅ AMZN → RMSE: 20.40 | MAE: 17.58

Training LSTM for NFLX...
  ✅ NFLX → RMSE: 3.53 | MAE: 2.88

Training LSTM for GOOGL...
  ✅ GOOGL → RMSE: 12.89 | MAE: 10.60

✅ LSTM training complete for all stocks!


## Step 6 — Save All Predictions

In [7]:
os.makedirs("data/predictions", exist_ok=True)

for ticker in TICKERS:
    np.save(f"data/predictions/{ticker}_lr_pred.npy",   lr_predictions[ticker])
    np.save(f"data/predictions/{ticker}_lstm_pred.npy", lstm_predictions[ticker])

print("✅ All predictions saved → data/predictions/")
print(f"   Total files: {len(TICKERS) * 2}  ({len(TICKERS)} LR + {len(TICKERS)} LSTM)")

✅ All predictions saved → data/predictions/
   Total files: 10  (5 LR + 5 LSTM)


## ✅ Notebook 4 Summary

**Models trained:**
- ✅ Linear Regression — fast, simple baseline
- ✅ LSTM — deep learning, learns sequential patterns

**Both models trained on all 5 FAANG stocks**

**Predictions saved:**
- `data/predictions/{TICKER}_lr_pred.npy`
- `data/predictions/{TICKER}_lstm_pred.npy`

**Next →** `05_evaluation_and_viz.ipynb`